In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install -q transformers datasets evaluate accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00


In this model, we use DeBERTa (Decoding-enhanced BERT with Disentangled Attention), a pretrained transformer model developed by Microsoft. Compared to earlier transformer models, DeBERTa uses a different attention mechanism that helps it understand the relationship between words and their positions more effectively. This allows the model to better capture the meaning and context of the question and answer options.

The multiple-choice problem is converted into a pairwise binary classification task. Each question is combined with each of its five answer options to create five question–option pairs. The correct option is assigned a label of 1, while the incorrect options are assigned a label of 0. The model is then fine-tuned to predict the probability that each option is the correct answer. During inference, the option with the highest probability is selected as the final answer, and the top three probabilities are used to generate the Top-3 predictions for calculating the MAP@3 score.

In [3]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
import wandb

from kaggle_secrets import UserSecretsClient

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    TrainerCallback
)

from datasets import Dataset

In [4]:
#W &B Login

from kaggle_secrets import UserSecretsClient
import wandb

user_secrets = UserSecretsClient()

api_key = user_secrets.get_secret("WANDB_API_KEY")

wandb.login(key=api_key)

wandb.init(
    entity="25ds1000066-dl-genai-project",
    project="25ds1000066-t22026",
    name="Model6_DeBERTa",
    config={
        "model":"DeBERTa-v3-small",
        
        "architecture":"Pairwise Classification",

        "optimizer":"AdamW",

        "learning_rate":2e-5,

        "epochs":5

    }

)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260723_095934-j72lhnq5
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run Model6_DeBERTa
wandb: ⭐️ View project at https://wandb.ai/25ds1000066-dl-genai-project/25ds1000066-t22026
wandb: 🚀 View run at https://wandb.ai/25ds1000066-dl-genai-project/25ds1000066-t22026/runs/j72lhnq5


In [5]:
# Configuration

MODEL_NAME = "microsoft/deberta-v3-small"

MAX_LENGTH = 320

BATCH_SIZE = 8

LR = 2e-5

EPOCHS = 5

SEED = 42

# Select Training Device
# -------------------------

device = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)

print(device)

cuda


In [6]:
# Reading the Dataset

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train.shape)
print(test.shape)

(2000, 8)
(500, 7)


In [7]:
# Split Dataset into Training and Validation Sets
# -------------------------------------------------

from sklearn.model_selection import train_test_split

# Split the original training data into
# 80% training and 20% validation

train_split, valid_split = train_test_split(
    train,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

print("Training Questions :", len(train_split))
print("Validation Questions :", len(valid_split))

Training Questions : 1600
Validation Questions : 400


In [8]:
# Create Pairwise Training Dataset
# ---------------------------------

# Convert each question into five question-option pairs.
# The correct option receives label 1, and the remaining
# four options receive label 0.

def create_pairwise_dataset(dataframe):

    pairwise_data = []

    for _, row in dataframe.iterrows():

        prompt = str(row["prompt"])

        correct_answer = row["answer"]

        # Create one training example for each option
        for option in ["A", "B", "C", "D", "E"]:

            pairwise_data.append({

                "question": prompt,

                "option": str(row[option]),

                # Label = 1 for the correct answer
                # Label = 0 for incorrect answers
                "label": int(option == correct_answer),

                # Store for later evaluation
                "correct_option": correct_answer,
                "option_name": option

            })

    return pd.DataFrame(pairwise_data)


# Create Pairwise Training and Validation Data
# ----------------------------------------------

deberta_train_pairwise = create_pairwise_dataset(train_split)

deberta_valid_pairwise = create_pairwise_dataset(valid_split)

print("Training Samples :", len(deberta_train_pairwise))

print("Validation Samples :", len(deberta_valid_pairwise))

display(deberta_train_pairwise.head())

Training Samples : 8000
Validation Samples : 2000


,question,option,label,correct_option,option_name
0,Pick the best possible answer: What is the pro...,Inflaton,1,A,A
1,Pick the best possible answer: What is the pro...,Quanta,0,A,B
2,Pick the best possible answer: What is the pro...,Scalar,0,A,C
3,Pick the best possible answer: What is the pro...,Metric,0,A,D
4,Pick the best possible answer: What is the pro...,Conformal cyclic cosmology,0,A,E


In [9]:
# Load DeBERTa Tokenizer
#--------------------------

# Load the tokenizer corresponding to the pretrained
# DeBERTa model.
deberta_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer Loaded Successfully")

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Tokenizer Loaded Successfully


In [10]:
# Tokenize Question-Option Pairs
# --------------------------------

# Convert each question-option pair into token IDs
# and attention masks that can be fed into RoBERTa.

def tokenize_function(examples):

    return deberta_tokenizer(

        examples["question"],
        examples["option"],

        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH

    )

In [11]:
# Convert Pandas DataFrames to Hugging Face Datasets
#-----------------------------------------------------

deberta_train_dataset = Dataset.from_pandas(
    deberta_train_pairwise
)

deberta_valid_dataset = Dataset.from_pandas(
    deberta_valid_pairwise
)

In [12]:
# Apply Tokenization
# --------------------

deberta_train_dataset = deberta_train_dataset.map(
    tokenize_function,
    batched=True
)

deberta_valid_dataset = deberta_valid_dataset.map(
    tokenize_function,
    batched=True
)



Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [13]:
# Rename Label Column
# ---------------------

deberta_train_dataset = deberta_train_dataset.rename_column(
    "label",
    "labels"
)

deberta_valid_dataset = deberta_valid_dataset.rename_column(
    "label",
    "labels"
)


# Keep Only Required Columns
# ---------------------------

deberta_train_dataset.set_format(

    type="torch",

    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]

)

deberta_valid_dataset.set_format(

    type="torch",

    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]

)

print(deberta_train_dataset)
print(deberta_valid_dataset)

Dataset({
    features: ['question', 'option', 'labels', 'correct_option', 'option_name', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 8000
})
Dataset({
    features: ['question', 'option', 'labels', 'correct_option', 'option_name', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2000
})


In [14]:
# Load DeBERTa Model
#-----------------------

# Binary classifier
# Label 1 = Correct Answer
# Label 0 = Incorrect Answer

deberta_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)


deberta_model.to(device)

print("DeBERTa Loaded Successfully")

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias       

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

DeBERTa Loaded Successfully


In [15]:
# Compute Evaluation Metrics
# ----------------------------

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, predictions)

    precision = precision_score(
        labels,
        predictions,
        average="binary",
        zero_division=0
    )

    recall = recall_score(
        labels,
        predictions,
        average="binary",
        zero_division=0
    )

    f1 = f1_score(
        labels,
        predictions,
        average="binary",
        zero_division=0
    )

    return {

        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1

    }

In [16]:
class WandbMetricsCallback(TrainerCallback):

    def on_log(self, args, state, control, logs=None, **kwargs):

        if logs is None:
            return

        metrics = {}

        # Training Loss
        if "loss" in logs:
            metrics["Training Loss"] = logs["loss"]

        # Validation Loss
        if "eval_loss" in logs:
            metrics["Validation Loss"] = logs["eval_loss"]

        # Validation Accuracy
        if "eval_accuracy" in logs:
            metrics["Validation Accuracy"] = logs["eval_accuracy"]

        if metrics:
            wandb.log(metrics)

In [17]:
# Training arguments
#---------------------

# Configure the Hugging Face Trainer
training_args = TrainingArguments(

    # Folder to save checkpoints
    output_dir="./deberta_checkpoints",

    # Training and Validation
    do_train=True,
    do_eval=True,

    # Evaluate after every epoch
    eval_strategy="epoch",

    # Save checkpoint after every epoch
    save_strategy="epoch",
    

    # Load the best checkpoint automatically
    load_best_model_at_end=True,

    # Use Validation F1 to determine the best model
    metric_for_best_model="f1",
    greater_is_better=True,

    # Keep only the best checkpoint
    save_total_limit=1,

    # Batch size
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    # Optimizer
    learning_rate=LR,
    weight_decay=0.01,

    # Epochs
    num_train_epochs=EPOCHS,

    # Logging
    logging_strategy="steps",
    logging_steps=50,

    # W&B
    report_to="wandb",
    run_name="Model6_DeBERTa",

    # Remove unnecessary columns automatically
    remove_unused_columns=True,

    # Mixed precision
    fp16=torch.cuda.is_available(),

    # Seed
    seed=SEED
)

In [18]:
print(deberta_model.config.torch_dtype)

`torch_dtype` is deprecated! Use `dtype` instead!


torch.float16


In [19]:
deberta_model = deberta_model.float()

print(next(deberta_model.parameters()).dtype)

torch.float32


In [20]:
# Create Hugging Face Trainer
# -----------------------------

trainer = Trainer(
    
    model=deberta_model,
    
    args=training_args,
    
    train_dataset=deberta_train_dataset,
    
    eval_dataset=deberta_valid_dataset,
    
    compute_metrics=compute_metrics,
    # Stop training if Validation F1 does not improve
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        ),
        WandbMetricsCallback()
    ]
)

In [21]:
# Train the DeBERTa Model
# ------------------------

print("Starting DeBERTa Training...\n")

trainer.train()

print("\nTraining Completed Successfully!")

Starting DeBERTa Training...



/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.007520,0.980887,0.800000,0.000000,0.000000,0.000000
2,0.890781,0.726811,0.871000,0.726115,0.570000,0.638655
3,0.593980,0.495149,0.914000,0.931818,0.615000,0.740964
4,0.408273,0.327479,0.932500,0.833753,0.827500,0.830615
5,0.293301,0.251638,0.952500,0.913279,0.842500,0.876463


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye


Training Completed Successfully!


In [22]:
# Generate Predictions on Validation Dataset
# --------------------------------------------

# Predict on the validation dataset

deberta_validation_output = trainer.predict(
    deberta_valid_dataset
)

# Extract logits

deberta_validation_logits = deberta_validation_output.predictions

print("Validation prediction shape:", deberta_validation_logits.shape)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Validation prediction shape: (2000, 2)


In [23]:
# Convert Logits to Probabilities
# ---------------------------------

import torch
import torch.nn.functional as F

# Convert logits to probabilities

deberta_validation_probabilities = F.softmax(

    torch.tensor(deberta_validation_logits),

    dim=1

).numpy()

print(deberta_validation_probabilities[:5])

[[0.99715877 0.00284125]
 [0.9962025  0.00379752]
 [0.9870433  0.0129566 ]
 [0.01422818 0.98577183]
 [0.726086   0.27391404]]


In [24]:
# Convert Pairwise Predictions to Question Predictions
# ------------------------------------------------------

deberta_validation_top1_predictions = []

deberta_validation_top3_predictions = []

for i in range(0, len(deberta_valid_pairwise), 5):

    # Probability that the option is the correct answer
    scores = deberta_validation_probabilities[i:i+5, 1]

    option_names = deberta_valid_pairwise.iloc[i:i+5]["option_name"].tolist()

    ranking = sorted(

        zip(option_names, scores),

        key=lambda x: x[1],

        reverse=True

    )

    deberta_validation_top1_predictions.append(

        ranking[0][0]

    )

    deberta_validation_top3_predictions.append(

        [x[0] for x in ranking[:3]]

    )

print(deberta_validation_top3_predictions[:5])

[['D', 'E', 'C'], ['E', 'D', 'B'], ['A', 'D', 'C'], ['B', 'C', 'E'], ['C', 'E', 'D']]


In [25]:
# Creating MAP@3 Function

# Function to calculate Average Precision at K (AP@K) for a single prediction
def apk(actual, predicted, k=3):

    if len(predicted) > k:                   # Keep only the top-k predictions
        predicted = predicted[:k]            

    for i, p in enumerate(predicted):        # Iterate through the predicted labels
        if p == actual:                      # If the correct answer is found, return the score
            return 1 / (i + 1)

    return 0                                 # Return 0 if the correct answer is not in the top-k predictions


# Function to calculate Mean Average Precision at K (MAP@K)
def mapk(actuals, predictions, k=3):
    
    # Compute the average AP@K score over the entire dataset
    return sum(apk(a, p, k) for a, p in zip(actuals, predictions)) / len(actuals)

In [26]:
# Calculate Validation Metrics
# ------------------------------

# Actual correct answers for the validation questions

deberta_actual_answers = valid_split["answer"].tolist()


# Accuracy

deberta_accuracy = accuracy_score(

    deberta_actual_answers,

    deberta_validation_top1_predictions

)


# Precision

deberta_precision = precision_score(

    deberta_actual_answers,

    deberta_validation_top1_predictions,

    average="macro"

)


# Recall

deberta_recall = recall_score(

    deberta_actual_answers,

    deberta_validation_top1_predictions,

    average="macro"

)


# F1 Score

deberta_f1 = f1_score(

    deberta_actual_answers,

    deberta_validation_top1_predictions,

    average="macro"

)


# MAP@3

deberta_map3 = mapk(

    deberta_actual_answers,

    deberta_validation_top3_predictions

)


# Print Results

print(f"Accuracy : {deberta_accuracy:.4f}")

print(f"Precision : {deberta_precision:.4f}")

print(f"Recall : {deberta_recall:.4f}")

print(f"F1 Score : {deberta_f1:.4f}")

print(f"MAP@3 : {deberta_map3:.4f}")

Accuracy : 0.9625
Precision : 0.9584
Recall : 0.9666
F1 Score : 0.9617
MAP@3 : 0.9762


In [27]:
# Log Final Validation Metrics to Weights & Biases
# -------------------------------------------------

wandb.log({

    "Accuracy": deberta_accuracy,

    "Precision": deberta_precision,

    "Recall": deberta_recall,

    "F1 Score": deberta_f1,

    "MAP@3": deberta_map3,

    "Kaggle Score": 0.75602

})

In [28]:
# Save the Fine-tuned RoBERTa Model
# -----------------------------------

SAVE_PATH = "/kaggle/working/deberta_model"

trainer.save_model(SAVE_PATH)

print("DeBERTa model saved successfully.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DeBERTa model saved successfully.


In [29]:
# Save the Tokenizer
# --------------------

deberta_tokenizer.save_pretrained(SAVE_PATH)

print("Tokenizer saved successfully.")



Tokenizer saved successfully.


In [30]:
# Save Validation Metrics
# ----------------------------

import json

deberta_metrics = {

    "Accuracy": float(deberta_accuracy),

    "Precision": float(deberta_precision),

    "Recall": float(deberta_recall),

    "F1 Score": float(deberta_f1),

    "MAP@3": float(deberta_map3)

}

with open(

    f"{SAVE_PATH}/metrics.json",

    "w"

) as file:

    json.dump(

        deberta_metrics,

        file,

        indent=4

    )

print("Metrics saved successfully.")

Metrics saved successfully.


In [31]:
# Finish Weights & Biases Run
# ----------------------------

wandb.finish()



wandb: uploading data; updating run metadata
wandb: uploading data
wandb: uploading history steps 111-112, summary, console lines 61-75
wandb: 
wandb: Run history:
wandb:            Accuracy ▁
wandb:            F1 Score ▁
wandb:        Kaggle Score ▁
wandb:               MAP@3 ▁
wandb:           Precision ▁
wandb:              Recall ▁
wandb:       Training Loss █▇▇█▇▇▇▇▇▇▇▇▆▆▇▆▅▄▄▄▄▃▃▄▃▂▃▂▃▂▂▂▁▂▁▂▂▁▂▁
wandb: Validation Accuracy ▁▄▆▇█
wandb:     Validation Loss █▆▃▂▁
wandb:       eval/accuracy ▁▄▆▇█
wandb:                 +20 ...
wandb: 
wandb: Run summary:
wandb:            Accuracy 0.9625
wandb:            F1 Score 0.96169
wandb:        Kaggle Score 0.75602
wandb:               MAP@3 0.97625
wandb:           Precision 0.95842
wandb:              Recall 0.96656
wandb:       Training Loss 0.2933
wandb: Validation Accuracy 0.9525
wandb:     Validation Loss 0.25164
wandb:       eval/accuracy 0.9525
wandb:                 +25 ...
wandb: 
wandb: 🚀 View run Model6_DeBERTa at: https://wandb.